In [124]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [125]:
### Cell 2 — Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.metrics import mean_squared_error
sns.set(style="darkgrid")

In [126]:
### Cell 3 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [127]:
### Cell 4 — Load Data
def read_file(filename):
    directory = '/content/drive/My Drive/210_capstone/final_datasets/full_hist_attr/'
    return pd.read_parquet(directory + filename + '.parquet')

In [128]:
df = read_file('full_data_18to23_asofFeb25_1')

In [129]:
### Cell 5 — Rename Columns
df = df.rename(columns={
    'Date':  'date',
    'County': 'county',
    'BEV':   'bev',
    'PHEV':  'phev',
    'FCEV':  'fcev',
    'Max_Daily_Electricity_Usage': 'max_daily_electricity',
    'Per_Capita_Personal_Income_Adjusted': 'per_cap_income',
    'Total': 'total_ev_charge'
})

## FEATURE ENGINEERING

In [130]:
### Cell 6 — Target Engineering (BEFORE split)
df['date'] = pd.to_datetime(df['date'])
df['max_elec_per_capita']     = df['max_daily_electricity'] / df['total_pop']
df['max_elec_per_capita_log'] = np.log(df['max_elec_per_capita'])

target = 'max_elec_per_capita_log'

print(df.shape)


(127078, 74)


In [131]:
# Monthly baseline from 2018
monthly_baseline_2018 = (df[df['date'].dt.year == 2018]
    .assign(month=lambda x: x['date'].dt.month)
    .groupby(['county', 'month'])
    .apply(lambda x: np.average(x['max_daily_electricity'], weights=x['total_pop']),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'baseline_mw_monthly_2018'}))

print(monthly_baseline_2018.head(20).to_string())

# Check statewide pop-weighted monthly baseline
statewide = (df[df['date'].dt.year == 2018]
    .assign(month=lambda x: x['date'].dt.month)
    .groupby('month')
    .apply(lambda x: np.average(x['max_daily_electricity'], weights=x['total_pop']),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'statewide_baseline_mw'}))

print("\nStatewide pop-weighted monthly baseline 2018:")
print(statewide.to_string(index=False))

     county  month  baseline_mw_monthly_2018
0   Alameda      1               1409.050922
1   Alameda      2               1435.616411
2   Alameda      3               1448.586942
3   Alameda      4               1410.874330
4   Alameda      5               1458.798573
5   Alameda      6               1624.530957
6   Alameda      7               1853.098624
7   Alameda      8               1741.385606
8   Alameda      9               1571.093310
9   Alameda     10               1408.985096
10  Alameda     11               1379.494960
11  Alameda     12               1374.451028
12   Alpine      1                  0.975642
13   Alpine      2                  1.068054
14   Alpine      3                  1.068804
15   Alpine      4                  1.061018
16   Alpine      5                  1.177404
17   Alpine      6                  1.320984
18   Alpine      7                  1.495994
19   Alpine      8                  1.391024

Statewide pop-weighted monthly baseline 2018:
 month  

In [132]:
ROLLING_SPECS = [
    ("cdd65_pop_roll5",       "cdd65_pop",  5,  "sum"),
    ("hdd65_pop_roll5",       "hdd65_pop",  5,  "sum"),
    ("tmax_k_pop_roll5_max",  "tmax_k_pop", 5,  "max"),
    ("tmax_k_pop_roll7_mean", "tmax_k_pop", 7,  "mean"),

]

def add_rolling_features(df):
    df = df.sort_values(["county", "date"]).copy()
    g = df.groupby("county", sort=False)
    for new_col, src_col, window, agg in ROLLING_SPECS:
        df[new_col] = g[src_col].transform(
            lambda x, w=window, a=agg: getattr(x.rolling(w, min_periods=1), a)()
        )
    return df

In [133]:
df = add_rolling_features(df)

In [134]:
df.columns

Index(['date', 'county', 'dpt_afternoon_k_pop', 'hdd65', 'wind_peak_ms_mean',
       'wind_low_ms_mean', 'spfh_peak_kgkg_pop', 'wind_peak_ms_pop', 'cdd75',
       'tavg_k', 'dpt_afternoon_k_mean', 'cloud_cover_pct_pop',
       'dpt_morning_k_mean', 'tmax_k_pop', 'cdd65_pop', 'cloud_cover_pct_mean',
       'tmin_k', 'hdd65_pop', 'dpt_morning_k_pop', 'wind_low_ms_pop',
       'trange_k', 'cdd65', 'tmin_k_pop', 'cdd75_pop', 'spfh_peak_kgkg_mean',
       'tmax_k', 'real_data_urma', 'staying_total', 'entering_total',
       'leaving_total', 'real_data_commuting', 'cuml_count', 'cuml_sq_foot',
       'cuml_utility_cap', 'cuml_dc_load', 'real_data_data_centers',
       'Total_Daily_Electricity_Usage', 'hour_of_max', 'max_daily_electricity',
       'Public Level 1', 'Shared Private Level 1', 'Public Level 2',
       'Shared Private Level 2', 'Public DC Fast', 'Shared Private DC Fast',
       'total_ev_charge', 'real_data_ev_charging', 'bev', 'phev', 'fcev',
       'real_data_ev_poplution', 'pe

In [135]:
### Dew Point Depression Feature
import numpy as np

def add_dpd_features(df):
    P = 101325  # standard pressure Pa
    w = df['spfh_peak_kgkg_pop']

    # specific humidity → vapor pressure
    e = (w * P) / (0.622 + w)

    # vapor pressure → dew point (K)
    dpt_c = 243.04 * np.log(e / 611.2) / (17.625 - np.log(e / 611.2))
    df['dpt_derived_k'] = dpt_c + 273.15

    # dew point depression (K) — large = dry, small = humid
    df['dpd_k'] = (df['tmax_k_pop'] - df['dpt_derived_k']).clip(lower=0)

    # rolling 5-day mean depression
    df = df.sort_values(['county', 'date'])
    df['dpd_k_roll5'] = (df
        .groupby('county')['dpd_k']
        .transform(lambda x: x.rolling(5, min_periods=1).mean()))

    return df

df = add_dpd_features(df)

#print(f"dpd_k stats:\n{df['dpd_k'].describe()}")

In [136]:
# Impute per_cap_income — county median, fall back to global median
global_median = df['per_cap_income'].median()

df['per_cap_income'] = (df.groupby('county')['per_cap_income']
    .transform(lambda x: x.fillna(x.median())))

df['per_cap_income'] = df['per_cap_income'].fillna(global_median)

print(df['per_cap_income'].isna().sum())  # should be 0

0


In [137]:
# Annual baseline — county level
baseline_2018 = (df[df['date'].dt.year == 2018]
    .groupby('county')
    .apply(lambda x: (x['max_daily_electricity'] / x['total_pop']).mean(),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'baseline_mw_per_capita_2018'}))

df = df.merge(baseline_2018, on='county', how='left')

# Monthly baseline — county x month level
monthly_baseline_2018 = (df[df['date'].dt.year == 2018]
    .assign(month=lambda x: x['date'].dt.month)
    .groupby(['county', 'month'])
    .apply(lambda x: np.average(x['max_daily_electricity'], weights=x['total_pop']),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'baseline_mw_monthly_2018'}))

df = df.assign(month=df['date'].dt.month).merge(
    monthly_baseline_2018, on=['county', 'month'], how='left')

print(df[['county', 'date', 'baseline_mw_per_capita_2018', 'baseline_mw_monthly_2018']].head(10))
print(f"\nNaNs: {df[['baseline_mw_per_capita_2018','baseline_mw_monthly_2018']].isna().sum().sum()}")

    county       date  baseline_mw_per_capita_2018  baseline_mw_monthly_2018
0  Alameda 2018-01-01                     0.000904               1409.050922
1  Alameda 2018-01-02                     0.000904               1409.050922
2  Alameda 2018-01-03                     0.000904               1409.050922
3  Alameda 2018-01-04                     0.000904               1409.050922
4  Alameda 2018-01-05                     0.000904               1409.050922
5  Alameda 2018-01-06                     0.000904               1409.050922
6  Alameda 2018-01-07                     0.000904               1409.050922
7  Alameda 2018-01-08                     0.000904               1409.050922
8  Alameda 2018-01-09                     0.000904               1409.050922
9  Alameda 2018-01-10                     0.000904               1409.050922

NaNs: 0


# THE SPLIT

In [138]:
# Run this before creating datasets — on df before the split
dow_map = {d: i for i, d in enumerate(df['day_of_week'].unique())}
df['day_of_week'] = df['day_of_week'].map(dow_map)

# Re-split
train_df = df[df['date'].dt.year <= 2021].copy()
val_df   = df[df['date'].dt.year == 2022].copy()
test_df  = df[df['date'].dt.year == 2023].copy()

In [139]:
# Split
train_df = df[df['date'].dt.year <= 2021].copy()
val_df   = df[df['date'].dt.year == 2022].copy()
test_df  = df[df['date'].dt.year == 2023].copy()

print(f"Train: {train_df.shape}  ({train_df['date'].dt.year.min()}–{train_df['date'].dt.year.max()})")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")


Train: (84738, 83)  (2018–2021)
Val:   (21170, 83)
Test:  (21170, 83)


## BUILDING THE TRANSFORMER MODEL

In [140]:
# ── Feature Groups ─────────────────────────────────────────────────────────────
WEATHER_TIME = [
    'tmax_k_pop', 'tmin_k_pop', 'trange_k',
    'cdd65_pop', 'hdd65_pop', 'cdd75_pop',
    'spfh_peak_kgkg_pop', 'wind_peak_ms_pop', 'dpd_k',
    'month', 'quarter', 'day_of_week', 'is_holiday'
]  # 13 features

ROLLING_TIME = [
    'cdd65_pop_roll5', 'hdd65_pop_roll5',
    'tmax_k_pop_roll5_max', 'tmax_k_pop_roll7_mean', 'dpd_k_roll5',
    'month', 'quarter', 'day_of_week', 'is_holiday'
]  # 9 features

GEO_NUMERIC =  [
    'total_pop',
    'per_cap_income',
    #'baseline_mw_per_capita_2018',   # county annual anchor
    'baseline_mw_monthly_2018'       # county x month seasonal anchor
] # 3 numeric + county embedding

INFRA_TIME = [
    'cuml_count', 'cuml_sq_foot', 'cuml_utility_cap', 'cuml_dc_load', 'bev',
    'month', 'quarter', 'day_of_week', 'is_holiday'
]  # 9 features

TARGET = 'max_elec_per_capita_log'

In [141]:
# ── Dataset ────────────────────────────────────────────────────────────────────
class ClimateDataset(Dataset):
    def __init__(self, df, scalers=None, county_map=None, fit=False):

        w = df[WEATHER_TIME].values.astype(np.float32)
        r = df[ROLLING_TIME].values.astype(np.float32)
        g = df[GEO_NUMERIC].values.astype(np.float32)
        i = df[INFRA_TIME].values.astype(np.float32)

        if fit:
            self.scalers = {
                'w': StandardScaler().fit(w),
                'r': StandardScaler().fit(r),
                'g': StandardScaler().fit(g),
                'i': StandardScaler().fit(i),
            }
            self.county_map = {c: idx for idx, c in enumerate(df['county'].unique())}
        else:
            self.scalers = scalers
            self.county_map = county_map

        self.w = self.scalers['w'].transform(w)
        self.r = self.scalers['r'].transform(r)
        self.g = self.scalers['g'].transform(g)
        self.i = self.scalers['i'].transform(i)

        self.county = df['county'].map(self.county_map).fillna(0).values.astype(np.int64)
        self.y = df[TARGET].values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.w[idx]),
            torch.tensor(self.r[idx]),
            torch.tensor(self.g[idx]),
            torch.tensor(self.i[idx]),
            torch.tensor(self.county[idx]),
            torch.tensor(self.y[idx])
        )

In [142]:
# ── Attention Block ────────────────────────────────────────────────────────────
class AttentionBlock(nn.Module):
    """Self-attention + residual + FFN + residual — one complete transformer block"""
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(embed_dim, num_heads,
                                            dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
        )
        self.norm2   = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # self-attention + residual
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        # ffn + residual
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


In [143]:
# ── Stream Module ──────────────────────────────────────────────────────────────
class StreamEncoder(nn.Module):
    """Project features → embed_dim, run 2 attention blocks, flatten"""
    def __init__(self, n_features, embed_dim=32, num_heads=4, dropout=0.1):
        super().__init__()
        self.proj   = nn.Linear(1, embed_dim)
        self.block1 = AttentionBlock(embed_dim, num_heads, dropout)
        self.block2 = AttentionBlock(embed_dim, num_heads, dropout)
        self.out_dim = n_features * embed_dim

    def forward(self, x):
        # x: (batch, n_features) → (batch, n_features, 1) → (batch, n_features, embed_dim)
        x = self.proj(x.unsqueeze(-1))
        x = self.block1(x)
        x = self.block2(x)
        return x.flatten(1)  # (batch, n_features * embed_dim)


In [144]:
# ── Full Model ─────────────────────────────────────────────────────────────────
class ClimateFEAT(nn.Module):
    def __init__(self, n_counties, embed_dim=32, num_heads=4, dropout=0.1):
        super().__init__()

        # Stream 1 — weather + time
        self.weather_enc = StreamEncoder(13, embed_dim, num_heads, dropout)  # → 416

        # Stream 2 — rolling + time
        self.rolling_enc = StreamEncoder(9, embed_dim, num_heads, dropout)   # → 288

        # Stream 3 — geography
        self.county_emb  = nn.Embedding(n_counties + 1, 8)
        self.geo_dense   = nn.Sequential(
            nn.Linear(3 + 8, 32), nn.ReLU(), nn.Dropout(dropout)
        )                                                                      # → 32

        # Phase 2 — cross-stream attention on streams 1+2+3
        # project each stream to same dim for cross attention
        cross_dim = 64
        self.w_to_cross = nn.Linear(416, cross_dim)
        self.r_to_cross = nn.Linear(288, cross_dim)
        self.g_to_cross = nn.Linear(32,  cross_dim)

        self.cross_attn  = AttentionBlock(cross_dim, num_heads=4, dropout=dropout)
        # 3 tokens of cross_dim each → flatten → 192

        # Stream 4 — infra + time (added after cross attention)
        self.infra_dense = nn.Sequential(
            nn.Linear(9, 32), nn.ReLU(), nn.Dropout(dropout)
        )                                                                      # → 32

        # Final head
        # cross output: 3 * 64 = 192, infra: 32
        self.head = nn.Sequential(
            nn.Linear(192 + 32, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 64),       nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, w, r, g, i, county):
        # Phase 1 — within-stream
        w_out = self.weather_enc(w)                              # (batch, 416)
        r_out = self.rolling_enc(r)                              # (batch, 288)

        county_emb = self.county_emb(county)                     # (batch, 8)
        g_out = self.geo_dense(torch.cat([g, county_emb], dim=1)) # (batch, 32)

        # Phase 2 — cross-stream attention on 1+2+3
        # stack as sequence of 3 tokens: (batch, 3, cross_dim)
        w_tok = self.w_to_cross(w_out).unsqueeze(1)              # (batch, 1, 64)
        r_tok = self.r_to_cross(r_out).unsqueeze(1)              # (batch, 1, 64)
        g_tok = self.g_to_cross(g_out).unsqueeze(1)              # (batch, 1, 64)

        cross = torch.cat([w_tok, r_tok, g_tok], dim=1)          # (batch, 3, 64)
        cross = self.cross_attn(cross)                            # (batch, 3, 64)
        cross = cross.flatten(1)                                  # (batch, 192)

        # Stream 4 — infra added after cross attention
        i_out = self.infra_dense(i)                              # (batch, 32)

        # Final head
        out = self.head(torch.cat([cross, i_out], dim=1))
        return out.squeeze(-1)



In [145]:
# ── Training Loop ──────────────────────────────────────────────────────────────
best_val_loss  = float('inf')
patience_count = 0
PATIENCE       = 15

for epoch in range(150):
    model.train()
    train_losses = []
    for w, r, g, i, county, y in train_loader:
        w, r, g, i, county, y = [t.to(device) for t in [w, r, g, i, county, y]]
        optimizer.zero_grad()
        loss = criterion(model(w, r, g, i, county), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for w, r, g, i, county, y in val_loader:
            w, r, g, i, county, y = [t.to(device) for t in [w, r, g, i, county, y]]
            val_losses.append(criterion(model(w, r, g, i, county), y).item())

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    scheduler.step(val_loss)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_climate_feat.pt')
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(torch.load('best_climate_feat.pt'))
print(f"\nBest val loss: {best_val_loss:.4f}")

Epoch   0 | train 0.1763 | val 0.0192
Epoch   5 | train 0.1730 | val 0.0248
Epoch  10 | train 0.1708 | val 0.0222
Epoch  15 | train 0.1697 | val 0.0202
Epoch  20 | train 0.1708 | val 0.0211
Early stopping at epoch 21

Best val loss: 0.0186


In [146]:
model.eval()
all_preds = []

with torch.no_grad():
    for w, r, g, i, county, y in val_loader:
        w, r, g, i, county = [t.to(device) for t in [w, r, g, i, county]]
        preds = model(w, r, g, i, county)
        all_preds.append(preds.cpu().numpy())

preds_log = np.concatenate(all_preds)
val_df['preds_mwh_ft2'] = np.exp(preds_log) * val_df['total_pop']
val_df['actual_mwh']    = np.exp(val_df[TARGET]) * val_df['total_pop']

ft2_rmse = np.sqrt(mean_squared_error(val_df['actual_mwh'], val_df['preds_mwh_ft2']))
w       = val_df['total_pop'].values
wrmse   = np.sqrt(np.sum(w * (val_df['actual_mwh'] - val_df['preds_mwh_ft2'])**2) / np.sum(w))
wmean   = np.average(val_df['actual_mwh'], weights=w)
pop_wtd = 100 * wrmse / wmean

print(f"ClimateFEAT Transformer v2:")
print(f"  RMSE MWh:        {ft2_rmse:,.0f}")
print(f"  Pop-wtd RMSE %:  {pop_wtd:.1f}%")

# LA monthly bias
val_df['residual_mwh'] = val_df['actual_mwh'] - val_df['preds_mwh_ft2']
la = val_df[val_df['county'] == 'Los Angeles'].copy()
la['month'] = la['date'].dt.month
print("\nLA Monthly Bias:")
print(la.groupby('month').agg(
    bias_mwh=('residual_mwh', 'mean'),
    rmse_mwh=('residual_mwh', lambda x: np.sqrt((x**2).mean()))
).round(1).to_string())

ClimateFEAT Transformer v2:
  RMSE MWh:        168
  Pop-wtd RMSE %:  14.8%

LA Monthly Bias:
       bias_mwh  rmse_mwh
month                    
1         529.8     854.3
2         357.7     934.7
3         115.7     973.6
4         245.3     907.8
5          23.3     827.9
6        -253.2     703.4
7       -1498.9    1625.4
8       -1045.1    1168.7
9       -1108.8    1583.4
10       -332.4     829.0
11       -572.9    1056.4
12      -1085.0    1306.0


In [148]:
pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.5/813.5 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.1 MB/s eta 0:00:00


In [149]:
import mlflow
import joblib
import os
from datetime import datetime

run_ts        = datetime.now().strftime('%Y%m%d_%H%M')
model_version = 'v1'
model_name    = f'climatefeat_transformer_{model_version}_{run_ts}'

MODEL_DIR = '/content/drive/My Drive/210_capstone/MLFlow/models'
PRED_DIR  = '/content/drive/My Drive/210_capstone/MLFlow/predictions'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR,  exist_ok=True)

mlflow.set_tracking_uri(f'sqlite:///{MODEL_DIR}/mlflow.db')
mlflow.set_experiment('climate-feat-transformer')

with mlflow.start_run(run_name=model_name):

    # ── Params ────────────────────────────────────────────────────────────
    mlflow.log_param('model_version',   model_version)
    mlflow.log_param('embed_dim',       32)
    mlflow.log_param('num_heads',       4)
    mlflow.log_param('dropout',         0.1)
    mlflow.log_param('batch_size',      512)
    mlflow.log_param('lr',              3e-4)
    mlflow.log_param('weight_decay',    1e-3)
    mlflow.log_param('n_weather_feats', len(WEATHER_TIME))
    mlflow.log_param('n_rolling_feats', len(ROLLING_TIME))
    mlflow.log_param('n_geo_feats',     len(GEO_NUMERIC))
    mlflow.log_param('n_infra_feats',   len(INFRA_TIME))
    mlflow.log_param('best_epoch',      epoch)

    # ── Metrics ───────────────────────────────────────────────────────────
    mlflow.log_metric('best_val_loss',   best_val_loss)
    mlflow.log_metric('rmse_mwh',        ft2_rmse)
    mlflow.log_metric('pop_wtd_rmse_pct', pop_wtd)

    # ── Save model ────────────────────────────────────────────────────────
    torch.save(model.state_dict(), f'{MODEL_DIR}/{model_name}.pt')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}.pt')

    # ── Save scalers ──────────────────────────────────────────────────────
    joblib.dump(train_ds.scalers,    f'{MODEL_DIR}/{model_name}_scalers.pkl')
    joblib.dump(train_ds.county_map, f'{MODEL_DIR}/{model_name}_county_map.pkl')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}_scalers.pkl')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}_county_map.pkl')

    # ── Save predictions ──────────────────────────────────────────────────
    pred_df = val_df[['county', 'date', 'total_pop', 'actual_mwh',
                       'preds_mwh_ft2', 'residual_mwh']].copy()
    pred_df.to_parquet(f'{PRED_DIR}/{model_name}_preds_val.parquet', index=False)
    mlflow.log_artifact(f'{PRED_DIR}/{model_name}_preds_val.parquet')

    print(f'MLflow run logged — climate-feat-transformer / {model_name}')
    print(f'  best_val_loss:    {best_val_loss:.4f}')
    print(f'  rmse_mwh:         {ft2_rmse:,.0f}')
    print(f'  pop_wtd_rmse_pct: {pop_wtd:.1f}%')

2026/03/02 22:06:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/02 22:06:50 INFO mlflow.store.db.utils: Updating database tables
2026/03/02 22:06:52 INFO mlflow.tracking.fluent: Experiment with name 'climate-feat-transformer' does not exist. Creating a new experiment.


MLflow run logged — climate-feat-transformer / climatefeat_transformer_v1_20260302_2206
  best_val_loss:    0.0186
  rmse_mwh:         168
  pop_wtd_rmse_pct: 14.8%
